# Lab 6 演示：层级数据可视化（Trees and Treemaps）

本 Notebook 只整理课堂演示的 **Task 1–15**，不包含后面的 GDP Assignment。Python 部分负责把扁平 CSV 转换成层级 JSON；D3 部分应复制到 `lab6/index.html` 与 `lab6/lab6.js` 后，在本地服务器或 GitHub Pages 中查看。

## Task 1：认识扁平的层级数据

CSV 的每一行是一座城市；`root → continent → country → region → city` 这几列共同描述其在树中的位置，`population_thousands` 是叶节点的数值。

In [3]:
from pathlib import Path
import pandas as pd

# 兼容从项目根目录或 lab6/ 文件夹启动 Jupyter 的两种情况。
project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

csv_path = project_root / 'data' / 'lab6_small_hierarchy.csv'
if not csv_path.exists():
    raise FileNotFoundError(
        f'找不到 {csv_path}。请先将课程提供的 lab6_small_hierarchy.csv 放进 data/ 文件夹。'
    )

df = pd.read_csv(csv_path)
display(df.head())
print('列名：', df.columns.tolist())

,root,continent,country,region,city,population_thousands
0,World,North America,USA,California,Los Angeles,3900
1,World,North America,USA,California,San Francisco,870
2,World,North America,USA,New York,New York City,8400
3,World,North America,Canada,Ontario,Toronto,2900
4,World,North America,Canada,British Columbia,Vancouver,675


列名： ['root', 'continent', 'country', 'region', 'city', 'population_thousands']


## Task 2：用 Python 转为层级 JSON

递归函数每次按当前层级分组；当只剩 `city` 时，建立包含城市名称与人口的叶节点。

In [4]:
import json

def build_hierarchy(dataframe, levels, value_column):
    """把指定的层级列递归转换为 D3 可用的 children 结构。"""
    if len(levels) == 1:
        # 最后一层是城市；每个城市保存用于面积编码的人口值。
        return [
            {'name': row[levels[0]], 'value': row[value_column]}
            for _, row in dataframe.iterrows()
        ]

    current_level = levels[0]
    children = []
    for value, group in dataframe.groupby(current_level, sort=True):
        children.append({
            'name': value,
            'children': build_hierarchy(group, levels[1:], value_column)
        })
    return children

hierarchy = {
    'name': 'World',
    'children': build_hierarchy(
        df, ['continent', 'country', 'region', 'city'], 'population_thousands'
    )
}

json_path = project_root / 'data' / 'lab6_small_hierarchy.json'
with open(json_path, 'w', encoding='utf-8') as file:
    json.dump(hierarchy, file, indent=2, ensure_ascii=False)

print(f'已输出：{json_path}')
hierarchy

已输出：e:\文档\STATS 401\homework\lab\stats401-labs\data\lab6_small_hierarchy.json


{'name': 'World',
 'children': [{'name': 'Asia',
   'children': [{'name': 'Japan',
     'children': [{'name': 'Kansai',
       'children': [{'name': 'Osaka', 'value': 2750}]},
      {'name': 'Kanto', 'children': [{'name': 'Tokyo', 'value': 14000}]}]},
    {'name': 'South Korea',
     'children': [{'name': 'Seoul',
       'children': [{'name': 'Seoul', 'value': 9500}]}]}]},
  {'name': 'Europe',
   'children': [{'name': 'France',
     'children': [{'name': 'Auvergne-Rhône-Alpes',
       'children': [{'name': 'Lyon', 'value': 520}]},
      {'name': 'Île-de-France',
       'children': [{'name': 'Paris', 'value': 2100}]}]},
    {'name': 'Germany',
     'children': [{'name': 'Bavaria',
       'children': [{'name': 'Munich', 'value': 1500}]},
      {'name': 'Berlin', 'children': [{'name': 'Berlin', 'value': 3700}]}]}]},
  {'name': 'North America',
   'children': [{'name': 'Canada',
     'children': [{'name': 'British Columbia',
       'children': [{'name': 'Vancouver', 'value': 675}]},
      

## Task 3：HTML 骨架与载入 JSON

将以下内容放在 `index.html`。D3 通过 `d3.json()` 读取刚生成的 JSON；请以 Live Server、`python -m http.server` 或 GitHub Pages 开启网页，直接双击 HTML 通常会被浏览器的安全策略阻挡。

```html
<div id="tree"></div>
<div id="treemap"></div>
<div id="tooltip" class="tooltip"></div>
<script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
<script src="lab6.js"></script>
```

```javascript
d3.json('../data/lab6_small_hierarchy.json').then(data => {
  console.log(data); // 检查读取到的原始嵌套资料
  drawDemo(data);
});
```

## Task 4–9：建立、绘制并折叠 Tree

下面的 `drawTree` 同时覆盖：Task 4 的 `d3.hierarchy()`、Task 5 的 `.sum()`、Task 6 的布局、Task 7 的连线、Task 8 的节点，以及 Task 9 的点击展开/折叠。

```javascript
function drawTree(data) {
  const width = 1000, height = 650;
  const root = d3.hierarchy(data).sum(d => d.value || 0); // Task 4–5
  const layout = d3.tree().size([height - 100, width - 250]); // Task 6
  const svg = d3.select('#tree').append('svg').attr('width', width).attr('height', height);
  const group = svg.append('g').attr('transform', 'translate(100,50)');

  function updateTree() {
    layout(root); // 每次点击后重新计算节点坐标
    group.selectAll('*').remove();

    // Task 7：root.links() 的每项都包含 source（父）和 target（子）。
    group.selectAll('.link').data(root.links()).join('path')
      .attr('class', 'link').attr('fill', 'none').attr('stroke', '#999')
      .attr('d', d3.linkHorizontal().x(d => d.y).y(d => d.x));

    // Task 8：根据 x、y 把每个节点群组移动到树上的正确位置。
    const nodes = group.selectAll('.node').data(root.descendants()).join('g')
      .attr('class', 'node').attr('transform', d => `translate(${d.y},${d.x})`);
    nodes.append('circle').attr('r', 6)
      .attr('fill', d => d.children ? 'steelblue' : 'orange');
    nodes.append('text').attr('x', 10).attr('dy', '0.35em').text(d => d.data.name);

    // Task 9：children 存可见子节点，_children 存被隐藏的子节点。
    nodes.style('cursor', 'pointer').on('click', (event, d) => {
      if (d.children) { d._children = d.children; d.children = null; }
      else { d.children = d._children; d._children = null; }
      updateTree();
    });
  }
  updateTree();
}
```

## Task 10–13：Treemap、颜色与 Tooltip

Treemap 用叶节点人口决定矩形面积；矩形颜色表示该城市所属洲别。以下代码实现 Task 10–13。

```javascript
function getContinent(d) { // Task 12：回溯至 root 的下一层
  let current = d;
  while (current.depth > 1) current = current.parent;
  return current.data.name;
}

function drawTreemap(data) {
  const width = 900, height = 550;
  const root = d3.hierarchy(data).sum(d => d.value || 0).sort((a, b) => b.value - a.value);
  d3.treemap().size([width, height]).paddingInner(2).paddingOuter(4)(root); // Task 10
  const color = d3.scaleOrdinal(['North America', 'Europe', 'Asia'], d3.schemeTableau10);
  const tooltip = d3.select('#tooltip');
  const svg = d3.select('#treemap').append('svg').attr('width', width).attr('height', height);

  const cell = svg.selectAll('.cell').data(root.leaves()).join('g') // Task 11：只画城市叶节点
    .attr('class', 'cell').attr('transform', d => `translate(${d.x0},${d.y0})`);
  cell.append('rect').attr('width', d => d.x1 - d.x0).attr('height', d => d.y1 - d.y0)
    .attr('fill', d => color(getContinent(d)));
  cell.append('text').attr('x', 5).attr('y', 18).text(d => d.data.name);

  // Task 13：鼠标移入显示资料，移动时让提示框跟随鼠标。
  cell.on('mouseover', (event, d) => tooltip.style('opacity', 1)
        .html(`<strong>${d.data.name}</strong><br>Population: ${d.value.toLocaleString()} thousand`))
    .on('mousemove', event => tooltip.style('left', `${event.pageX + 10}px`).style('top', `${event.pageY + 10}px`))
    .on('mouseout', () => tooltip.style('opacity', 0));
}

function drawDemo(data) { drawTree(data); drawTreemap(data); }
```

搭配的 CSS：
```css
.tooltip { position:absolute; opacity:0; pointer-events:none; background:white; border:1px solid #aaa; padding:8px 10px; border-radius:4px; }
```

## Task 14：缩放到某个 Treemap 节点

缩放需要保留所有节点（不只叶节点），并以被点击节点的边界作为新的比例尺 domain。以下为核心函数；实际完整版本还应加入“返回父节点”按钮。

```javascript
// x、y 为 d3.scaleLinear().range([0, width]) / range([0, height])；cells 是所有要更新的群组。
function zoomTo(d) {
  x.domain([d.x0, d.x1]);
  y.domain([d.y0, d.y1]);
  cells.transition().duration(600)
    .attr('transform', node => `translate(${x(node.x0)},${y(node.y0)})`);
  cells.select('rect').transition().duration(600)
    .attr('width', node => x(node.x1) - x(node.x0))
    .attr('height', node => y(node.y1) - y(node.y0));
}
// 例如：cells.on('click', (event, d) => zoomTo(d));
```

## Task 15：比较 Treemap 切分方法

层级和人口数不变，只替换 `.tile()` 即可观察空间组织差异：Squarify 尽量生成接近正方形的矩形；Binary 二分可用空间；SliceDice 按层级交替横向与纵向切分。

```javascript
const squarifyLayout = d3.treemap().tile(d3.treemapSquarify).size([900, 550]);
const binaryLayout = d3.treemap().tile(d3.treemapBinary).size([900, 550]);
const sliceDiceLayout = d3.treemap().tile(d3.treemapSliceDice).size([900, 550]);

// 对不同的 d3.hierarchy(data).sum(...) 副本分别应用其中一个 layout，再绘制即可比较。
```

---

到这里为止是演示内容。GDP hierarchy、两张 assignment treemap、图例与设计说明均刻意未在此 Notebook 中完成。